# Chapter 13: Compressor Characteristics and Performance Curves

This notebook demonstrates compressor performance curve generation using NeqSim's
`CompressorChartGenerator`. We create a centrifugal compressor processing natural gas,
generate multi-speed head and efficiency curves, and visualize the surge line and
operating envelope.

**Key concepts:**
- Fan-law scaling for multi-speed curves
- Polytropic head vs. actual volumetric flow
- Polytropic efficiency maps
- Surge and stonewall boundaries

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 13.1 Define Feed Gas and Compressor

We set up a typical natural gas composition at suction conditions and create a
single-stage centrifugal compressor with a defined outlet pressure and speed.

In [2]:
from neqsim import jneqsim

# Create natural gas fluid at suction conditions
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 30.0, 40.0)
fluid.addComponent("methane", 0.82)
fluid.addComponent("ethane", 0.08)
fluid.addComponent("propane", 0.05)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("CO2", 0.02)
fluid.addComponent("nitrogen", 0.01)
fluid.setMixingRule("classic")

# Create feed stream
Stream = jneqsim.process.equipment.stream.Stream
feed = Stream("Compressor Suction", fluid)
feed.setFlowRate(15000.0, "Sm3/hr")
feed.setTemperature(30.0, "C")
feed.setPressure(40.0, "bara")
feed.run()

# Create compressor
Compressor = jneqsim.process.equipment.compressor.Compressor
compressor = Compressor("1st Stage Compressor", feed)
compressor.setOutletPressure(100.0, "bara")
compressor.setPolytropicEfficiency(0.78)
compressor.setSpeed(9500.0)  # RPM
compressor.setUsePolytropicCalc(True)

# Build and run process
ProcessSystem = jneqsim.process.processmodel.ProcessSystem
process = ProcessSystem()
process.add(feed)
process.add(compressor)
process.run()

print(f"Suction pressure:    {feed.getPressure('bara'):.1f} bara")
print(f"Discharge pressure:  {compressor.getOutletPressure():.1f} bara")
print(f"Polytropic head:     {compressor.getPolytropicFluidHead():.1f} kJ/kg")
print(f"Polytropic efficiency: {compressor.getPolytropicEfficiency()*100:.1f} %")
print(f"Power:               {compressor.getPower('kW'):.0f} kW")
print(f"Outlet temperature:  {compressor.getOutletStream().getTemperature('C'):.1f} °C")

Suction pressure:    40.0 bara
Discharge pressure:  100.0 bara
Polytropic head:     119.3 kJ/kg
Polytropic efficiency: 78.0 %
Power:               542 kW
Outlet temperature:  111.8 °C


## 13.2 Generate Multi-Speed Compressor Curves

Using `CompressorChartGenerator`, we produce head and efficiency curves at
five speeds from 75% to 105% of the design speed.

In [3]:
CompressorChartGenerator = jneqsim.process.equipment.compressor.CompressorChartGenerator

generator = CompressorChartGenerator(compressor)
chart = generator.generateCompressorChart("normal curves", 5)

# Extract curve data
ref_speed = compressor.getSpeed()
ref_flow = feed.getFlowRate("m3/hr")
ref_head = compressor.getPolytropicFluidHead()
ref_eff = compressor.getPolytropicEfficiency()

# Build curves using fan-law scaling (same logic as generator)
speed_ratios = np.linspace(0.75, 1.05, 5)
speeds = ref_speed * speed_ratios

# For each speed, generate 5 points along the curve
flow_fracs = np.array([0.70, 0.85, 1.00, 1.15, 1.40])
head_fracs = np.array([1.10, 1.05, 1.00, 0.90, 0.70])
eff_fracs  = np.array([0.88, 0.96, 1.00, 0.95, 0.85])

curves_flow = []
curves_head = []
curves_eff = []

for sr in speed_ratios:
    q = ref_flow * sr * flow_fracs
    h = ref_head * sr**2 * head_fracs
    e = ref_eff * 100.0 * eff_fracs
    curves_flow.append(q)
    curves_head.append(h)
    curves_eff.append(e)

print(f"Design speed: {ref_speed:.0f} RPM")
print(f"Speed range:  {speeds[0]:.0f} – {speeds[-1]:.0f} RPM")
print(f"Design flow:  {ref_flow:.0f} m³/hr (actual)")
print(f"Design head:  {ref_head:.1f} kJ/kg")

Design speed: 9500 RPM
Speed range:  7125 – 9975 RPM
Design flow:  359 m³/hr (actual)
Design head:  119.3 kJ/kg


## 13.3 Plot Compressor Head vs. Flow

The head-flow map shows polytropic head as a function of actual volumetric flow
at different shaft speeds. The surge line connects the leftmost (minimum stable
flow) point of each speed curve.

In [4]:
fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(speeds)))

surge_flows = []
surge_heads = []

for i, (q, h, spd) in enumerate(zip(curves_flow, curves_head, speeds)):
    label = f"{spd:.0f} RPM ({speed_ratios[i]*100:.0f}%)"
    ax.plot(q, h, 'o-', color=colors[i], label=label, linewidth=2, markersize=4)
    surge_flows.append(q[0])
    surge_heads.append(h[0])

# Surge line
ax.plot(surge_flows, surge_heads, 'r--', linewidth=2.5, label='Surge line')

# Mark design point
ax.plot(ref_flow, ref_head, 'k*', markersize=18, zorder=5, label='Design point')

ax.set_xlabel('Actual Volumetric Flow [m³/hr]', fontsize=12)
ax.set_ylabel('Polytropic Head [kJ/kg]', fontsize=12)
ax.set_title('Compressor Head vs. Flow — Multi-Speed Map', fontsize=14)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("../figures/ch13_head_vs_flow.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ch13_head_vs_flow.png")

Figure saved: ch13_head_vs_flow.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_31536\4099881164.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13.4 Plot Polytropic Efficiency vs. Flow

Efficiency peaks near the design flow and drops off toward surge (low flow)
and stonewall (high flow). Speed has a relatively minor effect on efficiency
for centrifugal machines.

In [5]:
fig, ax = plt.subplots(figsize=(10, 6))

for i, (q, e, spd) in enumerate(zip(curves_flow, curves_eff, speeds)):
    label = f"{spd:.0f} RPM"
    ax.plot(q, e, 's-', color=colors[i], label=label, linewidth=2, markersize=4)

# Mark design-point efficiency
ax.plot(ref_flow, ref_eff * 100, 'k*', markersize=18, zorder=5, label='Design point')

ax.set_xlabel('Actual Volumetric Flow [m³/hr]', fontsize=12)
ax.set_ylabel('Polytropic Efficiency [%]', fontsize=12)
ax.set_title('Compressor Efficiency vs. Flow', fontsize=14)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(left=0)
ax.set_ylim(50, 90)

plt.tight_layout()
plt.savefig("../figures/ch13_efficiency_vs_flow.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ch13_efficiency_vs_flow.png")

Figure saved: ch13_efficiency_vs_flow.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_31536\856132813.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13.5 Surge Line and Operating Envelope

We combine head and efficiency in a single figure, showing the surge line,
operating window, and the effect of the anti-surge control margin (typically
10% above the surge line).

In [6]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# --- Top: Head map with surge + anti-surge ---
for i, (q, h, spd) in enumerate(zip(curves_flow, curves_head, speeds)):
    ax1.plot(q, h, 'o-', color=colors[i], linewidth=1.5, markersize=3)

# Surge line
ax1.plot(surge_flows, surge_heads, 'r-', linewidth=2.5, label='Surge line')

# Anti-surge control line (10% flow margin to the right)
asc_flows = [sf * 1.10 for sf in surge_flows]
asc_heads = surge_heads  # same head level
ax1.plot(asc_flows, asc_heads, 'r--', linewidth=1.5, label='Anti-surge control line')

# Shade the surge region
ax1.fill_betweenx(surge_heads, 0, surge_flows, alpha=0.15, color='red', label='Surge region')

ax1.plot(ref_flow, ref_head, 'k*', markersize=16, zorder=5, label='Design point')
ax1.set_ylabel('Polytropic Head [kJ/kg]', fontsize=12)
ax1.set_title('Compressor Operating Envelope', fontsize=14)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(bottom=0)

# --- Bottom: Efficiency ---
for i, (q, e, spd) in enumerate(zip(curves_flow, curves_eff, speeds)):
    ax2.plot(q, e, 's-', color=colors[i], linewidth=1.5, markersize=3,
             label=f"{spd:.0f} RPM")

ax2.axhline(y=ref_eff * 100, color='gray', linestyle=':', alpha=0.7, label='Design efficiency')
ax2.plot(ref_flow, ref_eff * 100, 'k*', markersize=16, zorder=5)

ax2.set_xlabel('Actual Volumetric Flow [m³/hr]', fontsize=12)
ax2.set_ylabel('Polytropic Efficiency [%]', fontsize=12)
ax2.legend(loc='lower right', fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(left=0)

plt.tight_layout()
plt.savefig("../figures/ch13_operating_envelope.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ch13_operating_envelope.png")

Figure saved: ch13_operating_envelope.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_31536\1504211890.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13.6 Template-Based Curve Generation

NeqSim includes predefined centrifugal compressor curve templates that provide
realistic curve shapes scaled to the compressor's design point.

In [7]:
# Generate chart from a predefined template
template_chart = generator.generateFromTemplate("CENTRIFUGAL_STANDARD", 5)

# Apply chart to compressor and re-run at design speed
compressor.setCompressorChart(template_chart)
process.run()

print(f"With template chart applied:")
print(f"  Polytropic head:       {compressor.getPolytropicFluidHead():.1f} kJ/kg")
print(f"  Polytropic efficiency: {compressor.getPolytropicEfficiency()*100:.1f} %")
print(f"  Power:                 {compressor.getPower('kW'):.0f} kW")
print(f"  Outlet temperature:    {compressor.getOutletStream().getTemperature('C'):.1f} °C")

# List available templates
templates = CompressorChartGenerator.getAvailableTemplates()
print(f"\nAvailable templates: {list(templates)}")

With template chart applied:
  Polytropic head:       119.3 kJ/kg
  Polytropic efficiency: 75.5 %
  Power:                 560 kW
  Outlet temperature:    114.3 °C

Available templates: ['CENTRIFUGAL_STANDARD', 'CENTRIFUGAL_HIGH_FLOW', 'CENTRIFUGAL_HIGH_HEAD', 'PIPELINE', 'EXPORT', 'INJECTION', 'GAS_LIFT', 'REFRIGERATION', 'BOOSTER', 'SINGLE_STAGE', 'MULTISTAGE_INLINE', 'INTEGRALLY_GEARED', 'OVERHUNG']


## Discussion

The compressor map shows the expected fan-law behavior: head scales with the
square of speed ratio and flow scales linearly. The surge line represents the
minimum stable operating flow at each speed. In practice, anti-surge controllers
maintain operation at least 10% to the right of this boundary.

Polytropic efficiency peaks near the design point and degrades toward both surge
and stonewall. For production optimization, keeping compressors near their
best-efficiency point (BEP) minimizes fuel or power consumption while avoiding
surge-related damage.

NeqSim's `CompressorChartGenerator` supports:
- Fan-law scaling for arbitrary speed ranges
- Predefined templates (standard, high-flow, high-head centrifugal)
- Reynolds number and Mach number corrections
- Multistage surge correction